# Nokontwerp voor het redrawen en wandstrijken van een drankblikje

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Toepassing: productie van een aluminium drankblikje

De nok drijft de ram met stempel aan die een **voorgevormd aluminium napje** redrawt en daarna door meerdere strijkringen duwt. Het maken van het napje in de cupper en het later trimmen, wassen, bedrukken en vernissen vallen buiten dit nokmechanisme.

1. **Aanloop en redraw (30°–90°, +20 mm):** de stempel neemt het napje gecontroleerd mee en verkleint de diameter. Hiervoor nemen we een constante proceskracht van **5 kN** aan.
2. **Wandstrijken (90°–220°, +100 mm):** de cup passeert opeenvolgende strijkringen. De wand wordt dunner en langer tot een bliklichaam van ongeveer 115 mm. De aangenomen proceskracht stijgt van **5 kN tot 25 kN**.
3. **Terugkeer en strippen (220°–340°, -120 mm):** de stempel keert terug en het bliklichaam wordt afgestroopt. Hiervoor nemen we **3 kN trekkracht** aan.

De waarden zijn een **conservatieve ontwerpschatting voor een enkelvoudige didactische pers**, geen universele productiedata. De werkelijke kracht moet finaal uit materiaaltoestand, begin- en eindwanddikte, reductie per strijkring, matrijswrijving en smering worden bepaald. Belangrijk is dat alle componenten hieronder consequent op dezelfde gekozen belasting worden berekend.

### Ontwerpspecificatie

<a id="specs">Bewegingsspecificatie:</a>

- van 30° tot 90°: heffing van +20 mm;
- van 90° tot 220°: heffing van +100 mm;
- van 220° tot 340°: terugkeer van -120 mm;
- van 340° tot 360° en van 0° tot 30°: stilstand voor producttransfer.

De equivalente bewegende massa van ram, stempel en volger wordt op **40 kg** geschat. De statische proceskrachten zijn:

- redraw: 5 kN druk;
- wandstrijken: lineair oplopend van 5 tot 25 kN druk;
- strippen: 3 kN trek.

Voor deze grotere slag wordt de cyclus vertraagd tot **1,0 s per omwenteling (60 rpm)**. Dit beperkt de inertiekracht en geeft meer tijd voor producttoevoer, smering en afvoer. Een echte productielijn gebruikt doorgaans meerdere stations en een gespecialiseerde pers; deze notebook onderzoekt bewust één nok-aangedreven station.

In [ ]:
# Specifications for one beverage-can body cycle

startangle1 = 30
endangle1   = 90
theta_seg1  = endangle1 - startangle1
startlift1  = 0
endlift1    = 20
motionlaw1  = 4  # cycloid

startangle2 = 90
endangle2   = 220
theta_seg2  = endangle2 - startangle2
startlift2  = 20
endlift2    = 120
motionlaw2  = 4  # cycloid

startangle3 = 220
endangle3   = 340
theta_seg3  = endangle3 - startangle3
startlift3  = 120
endlift3    = 0
motionlaw3  = 4  # cycloid

startangleload1 = 30
endangleload1   = 90
startload1      = 5000
endload1        = 5000

startangleload2 = 90
endangleload2   = 220
startload2      = 5000
endload2        = 25000

startangleload3 = 220
endangleload3   = 315
startload3      = -3000
endload3        = -3000

In [ ]:
# Simulation variables of cam:

dtheta = 0.01  # resolution of simulation, in degrees

theta_deg      = np.arange(0,360,dtheta)  # array of cam angles in degrees
theta          = theta_deg*np.pi/180      # array of cam angles in radians
lift           = np.zeros(len(theta))     # lift follower in mm
vel            = np.zeros(len(theta))     # velocity follower in mm/rad
vel_deg        = np.zeros(len(theta))     # velocity follower in mm/degree
acc            = np.zeros(len(theta))     # acceleration follower in mm/rad^2
acc_deg        = np.zeros(len(theta))     # acceleration follower in mm/degree^2
jerk           = np.zeros(len(theta))     # jerk follower in mm/rad^3
jerk_deg       = np.zeros(len(theta))     # jerk follower in mm/degree^3
ext_load       = np.zeros(len(theta))     # external load in N
pressure_angle = np.zeros(len(theta))     # pressure angle in radians

# Cam design parameters, as output of the analysis
base_radius     = []  # base circle radius in mm
follower_radius = []  # follower radius in mm
exc             = []  # excentricity in mm
mass            = []  # equivalent mass in kg
spring_constant = []  # spring constant in N/mm
spring_preload  = []  # spring preload in N
rpm             = []  # rotations per minute of the cam
omega           = []  # angular velocity of the cam in rad/s

# search space of possible R0 values [mm]:
R0_min          =   0
R0_max          = 500
R0_delta        =   0.1
R0_vec          = np.arange(R0_min, R0_max, R0_delta)
# maximum pressure angle corresponding to the evaluated R0s:
alpha_max       = np.zeros_like(R0_vec)

beta_vec        = np.arange(1, 360, 0.1)   # sampling resolution for rotation
rho_min         = np.zeros_like(beta_vec)  # minimal follower radius at each beta

# input parameters of several analysis functions:
beta            = []  # cam angle interval of the segment
startangle      = []  # start angle of the segment in degrees
endangle        = []  # end angle of the segment in degrees
startlift       = []  # start lift of the segment in mm
endlift         = []  # end lift of the segment in mm
startangleload  = []  # start angle of the load in degrees
endangleload    = []  # end angle of the load in degrees
startload       = []  # start load in N
endload         = []  # end load in N
motionlaw       = []  # choice of motion law:
    # 1: dwell,
	  # 2: minimal rms acceleration,
	  # 3: harmonic,
	  # 4: cycloid,
	  # 5: 5th degree poly,
	  # 6: 7th degree poly

In [ ]:
# compute motion functions:

# a motion segment runs in one periodic cycle
# from "startangle" till "endangle",
# and moves from "startlift" till "endlift",
# with a given "motionlaw", and
# with interpolation resolution "dtheta":

def addMotionSegment(startangle,endangle,startlift,endlift,motionlaw,dtheta):
    assert startangle - endangle < 0, 'End angle should be bigger than start angle'
    start_index = int(startangle/dtheta)
    end_index = int(endangle/dtheta)
    beta = endangle - startangle
    L0 = startlift
    L1 = endlift
    x = np.linspace(0, 1, (end_index - start_index))
    if motionlaw == 1:  # dwell
        assert L1 - L0 == 0, 'The given input does not represent a dwell'
        lift[start_index:end_index] = L0 * np.ones_like(x)
        vel_deg[start_index:end_index] = np.zeros_like(x)
        acc_deg[start_index:end_index] = np.zeros_like(x)
        jerk_deg[start_index:end_index] = np.zeros_like(x)

    elif motionlaw == 2:  # 3rd order polynomial (minimal rms acceleration)
        L = L1 - L0
        lift[start_index:end_index] = L * (3 * x**2 - 2 * x**3) + L0
        vel_deg[start_index:end_index] = L / beta * (6 * x - 6 * x**2)
        acc_deg[start_index:end_index] = L / beta**2 * (6 - 12 * x)
        jerk_deg[start_index:end_index] = L / beta**3 * (- 12 )* np.ones_like(x)

    elif motionlaw == 3:  # harmonische
        L = L1 - L0
        lift[start_index:end_index] = L * (1 - np.cos(np.pi * x)) / 2 + L0
        vel_deg[start_index:end_index] = L / beta * np.sin(np.pi * x) * np.pi / 2
        acc_deg[start_index:end_index] = L / beta**2 * np.cos(np.pi * x) * np.pi**2 / 2
        jerk_deg[start_index:end_index] = -L / beta**3 * np.sin(np.pi * x) * np.pi**3 / 2

    elif motionlaw == 4:  # volle cycloide
        L = L1 - L0
        lift[start_index:end_index] = L * (x - np.sin(2 * np.pi * x) / (2 * np.pi)) + L0
        vel_deg[start_index:end_index] = L / beta * (1 - np.cos(2 * np.pi * x))
        acc_deg[start_index:end_index] = 2 * np.pi * L / beta**2 * np.sin(2 * np.pi * x)
        jerk_deg[start_index:end_index] = 4 * np.pi **2 * L / beta**3 * np.cos(2 * np.pi * x)

    elif motionlaw == 5:  # 5th degree poly
        L = L1 - L0
        lift[start_index:end_index] = L0 + L * (6 * x**5 - 15 * x**4 + 10 * x**3)
        vel_deg[start_index:end_index] = L / beta * (30 * x**4 - 60 * x**3 + 30 * x**2)
        acc_deg[start_index:end_index] = L / beta**2 * (120 * x**3 - 180 * x**2 + 60 * x)
        jerk_deg[start_index:end_index] = L / beta**3 * (360 * x**2 - 360 * x + 60)

    elif motionlaw == 6:  # 7th degree poly
        L = L1 - L0
        lift[start_index:end_index] = L0 + L * (-20 * x**7 + 70 * x**6 - 84 * x**5 + 35 * x**4)
        vel_deg[start_index:end_index] = L / beta * (-140 * x**6 + 420 * x**5 - 420 * x**4 + 140 * x**3)
        acc_deg[start_index:end_index] = L / beta**2 * (-840 * x**5 + 2100 * x**4 - 1680 * x**3 + 420 * x**2)
        jerk_deg[start_index:end_index] = L / beta**3 * (-4200 * x**4 + 8400 * x**3 - 5040 * x**2 + 840 * x)

    vel = vel_deg*180/np.pi
    acc = acc_deg*(180/np.pi)**2
    jerk = jerk_deg*(180/np.pi)**3

    return lift, vel_deg, acc_deg,jerk_deg, vel, acc, jerk

def plotMotionLaw(t,lift,vel,acc, jerk):
    fig1, ax1 = plt.subplots(nrows=4,ncols=1,constrained_layout=True)
    fig1.suptitle("Bewegingswet")
    
    ax1[0].plot(t, lift)
    ax1[0].set_ylabel("Lift [mm]")
    ax1[0].set_xlim([0,360])
    
    ax1[1].plot(t, vel)
    ax1[1].set_ylabel(r"Vel $[\mathrm{mm}/\mathrm{deg}]$")
    ax1[1].set_xlim([0,360])
    
    ax1[2].plot(t, acc)
    ax1[2].set_ylabel(r"Acc $[\mathrm{mm}/\mathrm{deg}^2]$")
    ax1[2].set_xlabel('Theta [deg]')
    ax1[2].set_xlim([0,360])

    ax1[3].plot(t, jerk)
    ax1[3].set_ylabel(r"Jerk $[\mathrm{mm}/\mathrm{deg}^3]$")
    ax1[3].set_xlabel('Theta [deg]')
    ax1[3].set_xlim([0,360])


# Add all segments to the motion law:

# dwell from 0 to startangle1:
lift, vel_deg, acc_deg, jerk_deg, vel, acc, jerk = addMotionSegment(0,startangle1,0,0,1,dtheta)

# segment 1: rise
lift, vel_deg, acc_deg, jerk_deg, vel, acc, jerk = addMotionSegment(startangle1,endangle1,startlift1,endlift1,motionlaw1,dtheta)

# segment 2: rise
lift, vel_deg, acc_deg, jerk_deg, vel, acc, jerk = addMotionSegment(startangle2,endangle2,startlift2,endlift2,motionlaw2,dtheta)

# segment 3: fall
lift, vel_deg, acc_deg, jerk_deg, vel, acc, jerk = addMotionSegment(startangle3,endangle3,startlift3,endlift3,motionlaw3,dtheta)

# dwell from endangle3 to 360:
lift, vel_deg, acc_deg, jerk_deg, vel, acc, jerk = addMotionSegment(endangle3,360,0,0,1,dtheta)


# Plot the motion law
plotMotionLaw(theta_deg,lift,vel_deg,acc_deg, jerk_deg)

plt.show()

### Keuze van de bewegingswet

De fundamentele wet van het nokontwerp vereist continuïteit van positie, snelheid en versnelling. Bij het wandstrijken is het contact stijf en is de dunne aluminium wand gevoelig voor plooien, scheuren en diktevariatie. Een vloeiende bewegingswet is dus belangrijker dan een extreem korte cyclustijd.

Voor alle drie segmenten wordt een **cycloïde** gebruikt. Die heeft nul snelheid en nul versnelling aan de segmentgrenzen, een beperkte piekversnelling en is daardoor een robuuste keuze voor de grote slag van 120 mm. De ruk springt aan de aansluitingen; voor een zeer snelle industriële machine kan een 7e-orde polynoom of servoprofiel aantrekkelijker zijn, maar dan moeten de hogere piekversnelling en motorbelasting opnieuw worden gecontroleerd.

**Samenvattingstabel van de bewegingswetten** (Les 5; dimensieloos, $s(\tau)$ met $\tau \in [0,1]$):

![Samenvattingstabel van de bewegingswetten](../images/motion-law-summary.png)

De dimensionale pieken volgen uit de dimensieloze met $\dfrac{d^nS}{dt^n}\big|_{\max} = \dfrac{L}{\beta^n}\,\omega^n\,(\text{dimensieloze piek})$.

In [ ]:
# Comparison of the admissible motion laws (satisfying the fundamental law)
# dimensionless velocity, acceleration and jerk over tau in [0,1]

tau = np.linspace(0, 1, 500)

def motion_law_dimensionless(name, tau):
    if name == 'cycloid':
        v = 1 - np.cos(2*np.pi*tau)
        a = 2*np.pi*np.sin(2*np.pi*tau)
        j = 4*np.pi**2*np.cos(2*np.pi*tau)
    elif name == '5th order':
        v = 30*tau**4 - 60*tau**3 + 30*tau**2
        a = 120*tau**3 - 180*tau**2 + 60*tau
        j = 360*tau**2 - 360*tau + 60
    elif name == '7th order':
        v = -140*tau**6 + 420*tau**5 - 420*tau**4 + 140*tau**3
        a = -840*tau**5 + 2100*tau**4 - 1680*tau**3 + 420*tau**2
        j = -4200*tau**4 + 8400*tau**3 - 5040*tau**2 + 840*tau
    return v, a, j

candidate_laws = ['cycloid', '5th order', '7th order']

fig, ax = plt.subplots(nrows=3, ncols=1, constrained_layout=True)
fig.suptitle('Comparison of motion laws (dimensionless)')
for name in candidate_laws:
    v, a, j = motion_law_dimensionless(name, tau)
    ax[0].plot(tau, v, label=name)
    ax[1].plot(tau, a, label=name)
    ax[2].plot(tau, j, label=name)
ax[0].set_ylabel('Vel $v$ [-]')
ax[1].set_ylabel('Acc $a$ [-]')
ax[2].set_ylabel('Jerk $j$ [-]')
ax[2].set_xlabel(r'$\tau$ [-]')
for k in range(3):
    ax[k].set_xlim([0, 1])
    ax[k].grid()
ax[0].legend()
plt.show()

In [ ]:
# Dimensional peak values of the chosen cycloidal profile, per segment
cycletime = 1.0
rpm   = 60 / cycletime
omega = rpm * 2*np.pi / 60
mass  = 40

v_max, a_max, j_max = 2, 2*np.pi, 4*np.pi**2
segments = [
    ('redraw', endlift1 - startlift1, theta_seg1),
    ('wall ironing', endlift2 - startlift2, theta_seg2),
    ('return', endlift3 - startlift3, theta_seg3),
]

print('%-14s %7s %9s %14s %14s %12s' %
      ('segment', 'L[mm]', 'beta[deg]', 'a_peak[m/s^2]', 'jerk[m/s^3]', 'F_inert[N]'))
for name, L, beta_deg in segments:
    beta = beta_deg * np.pi/180
    L_m  = abs(L) / 1000
    a_pk = L_m / beta**2 * omega**2 * a_max
    j_pk = L_m / beta**3 * omega**3 * j_max
    F_in = mass * a_pk
    print('%-14s %7.1f %9.0f %14.1f %14.0f %12.0f' % (name, L, beta_deg, a_pk, j_pk, F_in))

In [ ]:
# Verification of the selected cycle time
T_rev = 60 / rpm
print('Speed %.0f rpm -> %.3f s per full revolution (360 deg).' % (rpm, T_rev))
print('Specified cycle time 1.0 s: %s.' % ('OK' if abs(T_rev-1.0) < 1e-6 else 'differs'))
active_arc = endangle3 - startangle1
print('Active motion arc %d deg takes %.3f s; transfer dwell takes %.3f s.' %
      (active_arc, active_arc/360*T_rev, (360-active_arc)/360*T_rev))

**Besluit.** De cycloïde voldoet aan de fundamentele wet en houdt de inertiële belasting beheersbaar bij de gekozen 60 rpm. Vooral de brede arbeidsslag van 130° is belangrijk: dezelfde 100 mm over een kleinere nokhoek zou de versnelling, contactkracht en het piekkoppel sterk verhogen. De terugkeer krijgt 120° zodat ook de zware ram gecontroleerd terugkomt. De resterende 50° dient als stilstand voor producttransfer.

In [ ]:
# utility functions for external load

def addLoadSegment(startangleload,endangleload,startload,endload,dtheta):
    assert startangleload - endangleload < 0, 'End angle should be bigger than start angle'
    start_index = int(startangleload/dtheta)
    end_index = int(endangleload/dtheta)
    theta_segment = theta_deg[start_index:end_index]
    ext_load[start_index:end_index] = (endload - startload)/(endangleload - startangleload)*(theta_segment - startangleload) + startload

    return ext_load

def plotExternalLoad(theta_deg,ext_load):
    plt.figure()
    plt.plot(theta_deg, ext_load)
    plt.xlabel('Theta [deg]')
    plt.ylabel('External load [N]')
    plt.xlim([0,360])
    plt.title('External load')

In [ ]:
# utility functions for cam profile design (with eccentricity = 0!)

# relate sampled R0 values to pressure angle \alpha:
def Kloomok_Muffley_R0(beta,startlift,endlift,motionlaw):
    global R0_vec, alpha_max
    L0 = startlift
    L1 = endlift
    beta = beta * np.pi / 180
		# 100 samples in dimensionless "arc length" of motion law:
    x = np.arange(0, 1, 0.01)

    for i in range(len(R0_vec)):
        R0 = R0_vec[i]

        if motionlaw == 1:  # dwell
            assert L1 - L0 == 0, 'The given input does not represent a dwell'
            lift = L0 * np.ones_like(x)
            vel_ang = np.zeros_like(x)
            acc_ang = np.zeros_like(x)

        elif motionlaw == 2:  # minimal rms acceleration
            L = L1 - L0
            lift = L * (3 * x**2 - 2 * x**3) + L0
            vel_ang = L / beta * (6 * x - 6 * x**2)
            acc_ang = L / beta**2 * (6 - 12 * x)

        elif motionlaw == 3:  # harmonische
            L = L1 - L0
            lift = L * (1 - np.cos(np.pi * x)) / 2 + L0
            vel_ang = L / beta * np.sin(np.pi * x) * np.pi / 2
            acc_ang = L / beta**2 * np.cos(np.pi * x) * np.pi**2 / 2

        elif motionlaw == 4:  # volle cycloide
            L = L1 - L0
            lift = L * (x - np.sin(2 * np.pi * x) / (2 * np.pi)) + L0
            vel_ang = L / beta * (1 - np.cos(2 * np.pi * x))
            acc_ang = 2 * np.pi * L / beta**2 * np.sin(2 * np.pi * x)

        elif motionlaw == 5:  # 5th degree poly
            L = L1 - L0
            lift = L0 + L * (6 * x**5 - 15 * x**4 + 10 * x**3)
            vel_ang = L / beta * (30 * x**4 - 60 * x**3 + 30 * x**2)
            acc_ang = L / beta**2 * (120 * x**3 - 180 * x**2 + 60 * x)

        elif motionlaw == 6:  # 7th degree poly
            L = L1 - L0
            lift = L0 + L * (-20 * x**7 + 70 * x**6 - 84 * x**5 + 35 * x**4)
            vel_ang = L / beta * (-140 * x**6 + 420 * x**5 - 420 * x**4 + 140 * x**3)
            acc_ang = L / beta**2 * (-840 * x**5 + 2100 * x**4 - 1680 * x**3 + 420 * x**2)

        alpha = np.arctan2(vel_ang, R0 + lift)
        alpha_max[i] = np.max(np.abs(alpha)) * 180 / np.pi


# relate given R0 value to samples of curvature radii of motion law:
def Kloomok_Muffley_follower_radius(R0,span,startlift,endlift,motionlaw):
    global beta_vec, rho_min
    L0 = startlift
    L1 = endlift

    for i in range(len(beta_vec)):
        beta = beta_vec[i] * np.pi / 180
        theta = np.linspace(0, beta_vec[i], 100) * np.pi / 180
        x = theta / beta

        if motionlaw == 1:  # dwell
            assert L1 - L0 == 0, 'The given input does not represent a dwell'
            lift = L0 * np.ones_like(x)
            vel_ang = np.zeros_like(x)
            acc_ang = np.zeros_like(x)

        elif motionlaw == 2:  # minimal rms acceleration
            L = L1 - L0
            lift = L * (3 * x**2 - 2 * x**3) + L0
            vel_ang = L / beta * (6 * x - 6 * x**2)
            acc_ang = L / beta**2 * (6 - 12 * x)

        elif motionlaw == 3:  # harmonische
            L = L1 - L0
            lift = L * (1 - np.cos(np.pi * x)) / 2 + L0
            vel_ang = L / beta * np.sin(np.pi * x) * np.pi / 2
            acc_ang = L / beta**2 * np.cos(np.pi * x) * np.pi**2 / 2

        elif motionlaw == 4:  # volle cycloide
            L = L1 - L0
            lift = L * (x - np.sin(2 * np.pi * x) / (2 * np.pi)) + L0
            vel_ang = L / beta * (1 - np.cos(2 * np.pi * x))
            acc_ang = 2 * np.pi * L / beta**2 * np.sin(2 * np.pi * x)

        elif motionlaw == 5:  # 5th degree poly
            L = L1 - L0
            lift = L0 + L * (6 * x**5 - 15 * x**4 + 10 * x**3)
            vel_ang = L / beta * (30 * x**4 - 60 * x**3 + 30 * x**2)
            acc_ang = L / beta**2 * (120 * x**3 - 180 * x**2 + 60 * x)

        elif motionlaw == 6:  # 7th degree poly
            L = L1 - L0
            lift = L0 + L * (-20 * x**7 + 70 * x**6 - 84 * x**5 + 35 * x**4)
            vel_ang = L / beta * (-140 * x**6 + 420 * x**5 - 420 * x**4 + 140 * x**3)
            acc_ang = L / beta**2 * (-840 * x**5 + 2100 * x**4 - 1680 * x**3 + 420 * x**2)

        rho = ((R0 + lift)**2 + vel_ang**2)**(3/2) / ((R0 + lift)**2 + 2 * vel_ang**2 - (R0 + lift) * acc_ang)
        rho_min[i] = np.min(np.abs(rho))

    plt.figure()
    plt.plot(beta_vec, rho_min)
    plt.axvline(x=span, color='b')
    plt.text(span, 0.35*R0, 'span of this segment: (%s)'%(span), color='b', ha='right', va='center', rotation='vertical')
    plt.grid()
    plt.xlabel(r'rotation (degrees)')
    plt.ylabel(r'$\rho$ (mm)')
    plt.xlim([0, 360])
    plt.show()

In [ ]:
# utility functions for pressure angle

def calculatePressureAngle(vel,exc,base_radius,follower_radius,lift):
    pressure_angle = np.arctan((vel-exc)/(np.sqrt((base_radius+follower_radius)**2-exc**2)+lift))
    return pressure_angle

def plotPressureAngle(theta_deg,pressure_angle):
    plt.figure()
    plt.plot(theta_deg, pressure_angle*180/np.pi)
    plt.xlabel('rotation [deg]')
    plt.ylabel('Pressure angle [deg]')
    plt.xlim([0,360])

In [ ]:
# utility functions for curvature radius

def calculateRadiusCurvature(base_radius,follower_radius,exc,theta,lift,vel,acc):
    d = np.sqrt((base_radius + follower_radius)**2 - exc**2) #Nortron, Kinematic and dynamics of Machinery eq. 8.31c
    lambda_ = -theta - np.arctan2(exc, d+lift) + np.pi/2 # Nortron, Kinematic and dynamics of Machinery eq. 8.34, adapted for vertical follower instead of a horizontal one (--> + pi/2)
    dlambda = -1 + exc/((d+lift)**2 + exc**2) * vel # derivative of lambda with repect to theta
    ddlambda = exc/((d+lift)**2 + exc**2)*acc - 2*exc*(d+lift)*vel**2/((d+lift)**2 + exc**2)**2 # 2nd derivative of lambda with repect to theta

    g = np.sqrt((d+lift)**2 + exc**2)
    dg = vel*(d+lift)/np.sqrt((d+lift)**2 + exc**2) # derivative of g with repect to theta
    ddg = (vel**2  + acc*(d + lift))/np.sqrt((d + lift)**2 + exc**2) - (vel**2*(d + lift)**2)/((d + lift)**2 + exc**2)**(3/2) # 2nd derivative of g with repect to theta

    h = np.sin(lambda_)
    dh = np.cos(lambda_)*dlambda # derivative of h with repect to theta
    ddh = -np.sin(lambda_)*dlambda**2 + np.cos(lambda_)*ddlambda # 2nd derivative of h with repect to theta
    f = np.cos(lambda_)
    df = -np.sin(lambda_)*dlambda #derivative of f with repect to theta
    ddf = -np.cos(lambda_)*dlambda**2 - np.sin(lambda_)*ddlambda # 2nd derivative of f with repect to theta

    dx = df*g + f*dg #derivative of x with repect to theta
    ddx = ddf*g + 2*df*dg + f*ddg # 2nd derivative of x with repect to theta
    dy = dh*g + h*dg # derivative of y with repect to theta
    ddy = ddh*g + 2*dh*dg + h*ddg # 2nd derivative of y with repect to theta

    roc_pitch = -(dx**2 + dy**2)**(3/2) / (dx*ddy - dy*ddx) # general forumula for the radius of curvature
    roc_cam = roc_pitch - follower_radius

    return roc_pitch, roc_cam

def plotRadiusCurvature(theta_deg,roc_pitch,roc_cam):
    inds = np.where(np.abs(np.diff(roc_pitch)/dtheta)<1e3)[0]
    maxi = max(10, max(roc_pitch[inds+1]))
    mini = min(-10, min(roc_pitch[inds+1]))
    plt.figure()
    plt.plot(theta_deg, roc_pitch)
    plt.plot(theta_deg, roc_cam)
    plt.xlabel('Theta [deg]')
    plt.ylabel('Radius of curvature [mm]')
    plt.xlim([0,360])
    plt.ylim([mini,maxi])
    plt.grid()
    plt.title('Radius of curvature')

def plotCamContour(base_radius,follower_radius,exc,theta,lift,pressure_angle,dtheta,):
    d = np.sqrt((base_radius + follower_radius)**2 - exc**2)
    lambda_ = -theta - np.arctan2(exc, d+lift) + np.pi/2
    xpitch = np.cos(lambda_)*np.sqrt((d+lift)**2 + exc**2)
    ypitch = np.sin(lambda_)*np.sqrt((d+lift)**2 + exc**2)

    xcam = xpitch + follower_radius*np.cos(-theta + pressure_angle + 3*np.pi/2)
    ycam = ypitch + follower_radius*np.sin(-theta + pressure_angle + 3*np.pi/2)

    pitch_curve = np.array([xpitch,ypitch])
    cam_curve = np.array([xcam,ycam])

    x_rolfol_center = exc  # x coordinate of the center of the roller of the follower
    y_rolfol_center = d + lift[0]  # y coordinate of the center of the roller of the follower
    x_rolfol = follower_radius * np.cos(theta) + x_rolfol_center  # x coordinate of the contour of the roller
    y_rolfol = follower_radius * np.sin(theta) + y_rolfol_center  # y coordinate of the contour of the roller

    min_distance_center_roller_to_bore = 2 * follower_radius
    height_bore = follower_radius

    width_block_on_follower = base_radius / 2
    height_block_on_follower = base_radius / 2

    x_boreleft = np.array([exc - 5, exc - 1, exc - 1, exc - 5])
    x_boreright = np.array([exc + 5, exc + 1, exc + 1, exc + 5])
    y_bore = np.array([d + max(lift) + min_distance_center_roller_to_bore] * 2 + [d + max(lift) + min_distance_center_roller_to_bore + height_bore] * 2)
    length_follower = y_bore[-1] - ypitch[0] - follower_radius + 0.2

    x_follower = np.array([x_rolfol_center, x_rolfol_center])
    y_follower = np.array([y_rolfol_center + follower_radius, y_rolfol_center + follower_radius + length_follower])

    x_block_on_follower = np.array([-width_block_on_follower, width_block_on_follower, width_block_on_follower, -width_block_on_follower]) + x_follower[0]
    y_block_on_follower = np.array([0, 0, height_block_on_follower, height_block_on_follower]) + y_follower[1]

    ref_line = np.array([[0, 0], [0, base_radius]])  # line, indicating the reference (= cam position = 0 degrees)

    x_pressangle = np.array([x_rolfol_center + follower_radius * np.sin(pressure_angle[0]), x_rolfol_center - follower_radius * np.sin(pressure_angle[0])])
    y_pressangle = np.array([y_rolfol_center - follower_radius * np.cos(pressure_angle[0]), y_rolfol_center + follower_radius * np.cos(pressure_angle[0])])

    plt.figure()
    plt.fill(cam_curve[0, ::int(1 / dtheta)], cam_curve[1, ::int(1 / dtheta)], 'b')
    plt.fill(x_rolfol[::int(1 / dtheta)], y_rolfol[::int(1 / dtheta)], 'g')
    plt.plot(pitch_curve[0, ::int(1 / dtheta)], pitch_curve[1, ::int(1 / dtheta)], '--r')
    plt.plot(0, 0, 'r+')
    plt.plot(x_pressangle, y_pressangle, 'r--')
    plt.plot(ref_line[0, :], ref_line[1, :], 'r--')
    plt.plot(x_boreleft, y_bore, 'r')
    plt.plot(x_boreright, y_bore, 'r')
    plt.plot(x_follower, y_follower, 'r')
    plt.fill(x_block_on_follower, y_block_on_follower, 'g')

    xlimits = [min(pitch_curve[0, :]), max(pitch_curve[0, :])]
    ylimits = [min(pitch_curve[1, :]), d + max(lift) + follower_radius + length_follower + height_block_on_follower]

    plt.xlim(np.array(xlimits) * 1.2)
    plt.ylim(np.array(ylimits) * 1.05)
    plt.gca().set_aspect('equal')
    plt.grid(True)
    plt.title('Cam contour [mm]')

def calculateForce(rpm,lift,spring_constant,spring_preload,pressure_angle,ext_load,mass,acc):
    omega = rpm*2*np.pi/60
    normalforce_spring = (lift*spring_constant+spring_preload)/np.cos(pressure_angle)
    normalforce_load = ext_load/np.cos(pressure_angle)
    normalforce_acc = mass*acc/1000*(omega**2)/np.cos(pressure_angle)
    normalforce_tot = normalforce_spring + normalforce_load + normalforce_acc
    force_x = normalforce_tot * np.sin(pressure_angle)
    force_y = normalforce_tot * np.cos(pressure_angle)

    return normalforce_tot,normalforce_acc,normalforce_load,normalforce_spring,force_x,force_y

def plotForces(theta_deg,normalforce_tot,normalforce_acc,normalforce_load,normalforce_spring,force_x,force_y):
    plt.figure()
    plt.plot(theta_deg,normalforce_tot)
    plt.plot(theta_deg,normalforce_acc,'--r')
    plt.plot(theta_deg,normalforce_load,':g')
    plt.plot(theta_deg,normalforce_spring,'-.m')
    plt.xlabel('Theta [deg]')
    plt.ylabel('Normal force [N]')
    plt.xlim([0,360])
    plt.legend(['total','acc','extload','spring'])
    plt.title('Normal force')

    plt.figure()
    plt.plot(theta_deg,force_x)
    plt.xlabel('Theta [deg]')
    plt.ylabel('Force x [N]')
    plt.xlim([0,360])
    plt.title('Force x')

    plt.figure()
    plt.plot(theta_deg,force_y)
    plt.xlabel('Theta [deg]')
    plt.ylabel('Force y [N]')
    plt.xlim([0,360])
    plt.title('Force y')

### Ontwerp van minimale steekcirkel $R_0$

In [ ]:
# parameter values for the needed motion segments

# a motion segment runs in one periodic cycle
# from "startangle" till "endangle",
# and moves from "startlift" till "endlift",
# with a given "motionlaw", and
# with interpolation resolution "dtheta":

# The approach was first described by M. Kloomok and R.V. Muffley,
# in Chapter 3 of Mabie, H.H., and Ocvirk, F.W., Mechanisms and Dynamics of Machinery, 1957

# segment 1:
Kloomok_Muffley_R0(theta_seg1,startlift1,endlift1,motionlaw1)

plt.plot(R0_vec, alpha_max)
plt.grid()
plt.title('Segment 1')
plt.xlabel('$R_0$ (mm)')
plt.ylabel(r'$\alpha$ (degree)')
plt.xlim([0, 500])
plt.show()

# segment 2:
Kloomok_Muffley_R0(theta_seg2,startlift2,endlift2,motionlaw2)

plt.figure()
plt.plot(R0_vec, alpha_max)
plt.grid()
plt.title('Segment 2')
plt.xlabel('$R_0$ (mm)')
plt.ylabel(r'$\alpha$ (degree)')
plt.xlim([0, 500])
plt.show()

# segment 3:
Kloomok_Muffley_R0(theta_seg3,startlift3,endlift3,motionlaw3)

plt.figure()
plt.plot(R0_vec, alpha_max)
plt.grid()
plt.title('Segment 3')
plt.xlabel('$R_0$ (mm)')
plt.ylabel(r'$\alpha$ (degree)')
plt.xlim([0, 500])
plt.show()

Uit de drukhoekanalyse kiezen we een **steekstraal $R_0 = 250$ mm**. Een grotere heffing maakt een grotere nok noodzakelijk: met een kleine steekcirkel zou de drukhoek en dus de zijdelingse kracht op stempel en geleidingen te groot worden. De gekozen straal is een compromis tussen compacte inbouw en een drukhoek onder 30°.

In [ ]:
R0 = 250  # pitch radius [mm] for the beverage-can cam

De steekstraal is toepassinggebonden. Bij een andere slag, timing of excentriciteit moeten drukhoek en ondersnijding opnieuw worden berekend.

### Ontwerp van straal $R_r$ van de volger

In [ ]:
# compute R_r for all segments:

Kloomok_Muffley_follower_radius(R0,theta_seg1,startlift1,endlift1,motionlaw1)
Kloomok_Muffley_follower_radius(R0,theta_seg2,startlift2,endlift2,motionlaw2)
Kloomok_Muffley_follower_radius(R0,theta_seg3,startlift3,endlift3,motionlaw3)

De rol moet groot genoeg zijn voor de hoge contactkracht, maar kleiner blijven dan de minimale kromtestraal om ondersnijding te vermijden. We kiezen **$R_r = 50$ mm**. Daarmee wordt de basiscirkel **$R_b = R_0-R_r = 200$ mm**. De grotere rol verlaagt de Hertz-contactspanning en lagerbelasting ten opzichte van de kleine referentievolger, terwijl de kromtestraalcontrole hieronder bepaalt of de geometrie geldig blijft.

In [ ]:
# Kinematic inputs for the beverage-can design
base_radius     = 200
follower_radius = 50
exc = 35  # provisional value; checked and optimized below

pressure_angle = calculatePressureAngle(vel, exc, base_radius, follower_radius, lift)
plotPressureAngle(theta_deg, pressure_angle)

roc_pitch, roc_cam = calculateRadiusCurvature(base_radius, follower_radius, exc, theta, lift, vel, acc)
plotRadiusCurvature(theta_deg, roc_pitch, roc_cam)
plotCamContour(base_radius, follower_radius, exc, theta, lift, pressure_angle, dtheta)

In [ ]:
# Undercutting check (Les 6):  rho_min > R_r
# rho_pitch = radius of curvature of the pitch curve. On convex parts (rho_pitch > 0) it
# must not become smaller than the roller radius R_r, otherwise a cusp / undercutting occurs.
finite  = np.abs(np.diff(roc_pitch) / dtheta) < 1e3      # drop the transition spikes
convex  = (roc_pitch[1:] > 0) & finite
rho_min = np.min(roc_pitch[1:][convex])
print('Minimum (convex) pitch-curve radius of curvature rho_min = %.2f mm' % rho_min)
print('Roller radius R_r                                        = %.1f mm' % follower_radius)
print('Undercutting: %s' % ('OK -- rho_min > R_r, no undercutting'
      if rho_min > follower_radius else 'FAIL -- rho_min <= R_r, undercutting!'))

### Optimalisatie van de excentriciteit voor het drankblikje

De grootste contactkracht treedt op tijdens de wandstrijkslag, waar de proceskracht tot 25 kN stijgt. Positieve excentriciteit verlaagt daar de drukhoek en dus de transversale kracht $T=N\sin\alpha$, maar verhoogt de drukhoek tijdens de lichter belaste terugkeer. We zoeken daarom een excentriciteit die de drukhoek tijdens de arbeidsslag verlaagt terwijl $|\alpha|<30°$ over de volledige cyclus blijft.

In [ ]:
# Coarse exploration of eccentricity
plt.figure()
for e in [0, 20, 40, 60, 80]:
    alpha = calculatePressureAngle(vel, e, base_radius, follower_radius, lift)
    plt.plot(theta_deg, alpha*180/np.pi, label='e = %d mm' % e)
plt.axhline( 30, color='r', ls='--')
plt.axhline(-30, color='r', ls='--')
plt.axvspan(startangle2, endangle2, color='grey', alpha=0.15)
plt.xlabel('Theta [deg]')
plt.ylabel('Pressure angle [deg]')
plt.xlim([0, 360])
plt.grid()
plt.legend()
plt.title('Pressure angle per eccentricity')
plt.show()

De excentriciteit verschuift de gunstige drukhoek naar de zwaar belaste heenbeweging. Een te grote excentriciteit is niet zinvol, omdat dan de terugkeer en de lagerreacties verslechteren. De volgende cel zoekt het optimum binnen de geometrisch mogelijke waarden.

In [ ]:
# Optimize eccentricity on pressure angle during wall ironing
rise = (theta_deg >= startangle2) & (theta_deg < endangle2)
exc_arr     = np.arange(0, 100, 0.25)
alpha_rise  = np.zeros_like(exc_arr)
alpha_total = np.zeros_like(exc_arr)
for i in range(len(exc_arr)):
    alpha = calculatePressureAngle(vel, exc_arr[i], base_radius, follower_radius, lift)
    alpha_rise[i]  = np.max(np.abs(alpha[rise])) * 180 / np.pi
    alpha_total[i] = np.max(np.abs(alpha)) * 180 / np.pi

allowed = alpha_total <= 30
exc_opt = exc_arr[allowed][np.argmin(alpha_rise[allowed])]
print('Optimal eccentricity: %.2f mm' % exc_opt)

plt.figure()
plt.plot(exc_arr, alpha_rise, label='wall-ironing stroke')
plt.plot(exc_arr, alpha_total, label='full rotation')
plt.axhline(30, color='r', ls='--')
plt.axvline(exc_opt, color='b')
plt.xlabel('Eccentricity [mm]')
plt.ylabel(r'max $|\alpha|$ [deg]')
plt.xlim([0, 100])
plt.grid()
plt.legend()
plt.title('Optimal eccentricity from pressure angle')
plt.show()

Het berekende optimum staat in de uitvoer hierboven. De definitieve keuze wordt verderop afgerond op een praktisch maakbare waarde en ook gecontroleerd op transversale kracht en wrijvingsarbeid.

In [ ]:
# generate external load

ext_load = addLoadSegment(startangleload1,endangleload1,startload1,endload1,dtheta)
ext_load = addLoadSegment(startangleload2,endangleload2,startload2,endload2,dtheta)
ext_load = addLoadSegment(startangleload3,endangleload3,startload3,endload3,dtheta)

plotExternalLoad(theta_deg,ext_load)

### Veerdimensionering

De nok kan de rolvolger alleen duwen. Een veer houdt daarom het contact tijdens de terugkeer in stand. Door de grotere bewegende massa en stripkracht moet zowel voorspanning als veerconstante toenemen. We kiezen eerst 2 kN voorspanning en bepalen vervolgens de minimale veerconstante uit $N\geq0$, met 25% veiligheidsmarge. In een echte blikjespers is een positieve terugloop via een groefnok, conjugate cam of krukmechanisme waarschijnlijk robuuster dan één zeer zware sluitveer.

In [ ]:
# Spring design (Les 6): choose the preload, then compute the spring rate so that N >= 0

mass  = 40                          # kg (equivalent mass)
rpm   = 60                         # rpm (cycle time 1.0 s)
omega = rpm * 2*np.pi / 60          # rad/s

F_inert = mass * acc/1000 * omega**2   # inertial force m*omega^2*S'' [N]

# 1) choose the preload F_v0
spring_preload = 2000.0              # N

# 2) minimum spring rate so that N >= 0:  k_v >= max( -(F_v0 + F_func + F_inert) / S )
mask  = lift > 1e-6
k_min = max(0.0, np.max(-(spring_preload + ext_load + F_inert)[mask] / lift[mask]))

# 3) safety margin so that contact is not lost 'just barely'
safety          = 1.25
spring_constant = k_min * safety
print('Preload F_v0          : %.0f N' % spring_preload)
print('Minimum spring rate   : %.2f N/mm' % k_min)
print('Chosen spring rate    : %.2f N/mm (margin %.0f%%)' % (spring_constant, (safety-1)*100))

# Calculate and plot the forces
[normalforce_tot,normalforce_acc,normalforce_load,normalforce_spring,force_x,force_y] = calculateForce(
    rpm, lift, spring_constant, spring_preload, pressure_angle, ext_load, mass, acc)
print('Minimum contact force : %.1f N  (> 0  =>  force closure OK)' % np.min(normalforce_tot))
print('Maximum contact force : %.1f N' % np.max(normalforce_tot))
plotForces(theta_deg,normalforce_tot,normalforce_acc,normalforce_load,normalforce_spring,force_x,force_y)

We vergelijken twee verschillende doelfuncties:
- de **piek** van $|T|$ over de volledige rotatie — maatgevend voor klemmen, de maximale spanning en het zwaarste slijtage-moment;
- de **wrijvingsarbeid** $\int |T|\,R\, d\theta$ per cyclus (in J, met $R(\theta)=R_0+f(\theta)$ de arm) — het verspilde vermogen $P=T\,R\,\omega$ geïntegreerd over een omwenteling, dus een maat voor de gemiddelde slijtage over de levensduur.

(De veer blijft op de eerder ontworpen waarden; we bekijken enkel het effect van de excentriciteit op de transversale kracht.)

In [ ]:
# Step 3: optimize on the transverse force T = N*sin(alpha) = force_x

exc_arr    = np.arange(0, 100, 0.25)
T_peak     = np.zeros_like(exc_arr)   # peak of |T| over the full rotation [N]
W_friction = np.zeros_like(exc_arr)   # friction work per cycle [J]
dtheta_rad = dtheta * np.pi / 180     # angular step in radians
R          = (R0 + lift) / 1000.0     # arm = pitch radius in m

for i in range(len(exc_arr)):
    alpha = calculatePressureAngle(vel, exc_arr[i], base_radius, follower_radius, lift)
    _, _, _, _, force_x, _ = calculateForce(rpm, lift, spring_constant, spring_preload,
                                            alpha, ext_load, mass, acc)
    T_peak[i]     = np.max(np.abs(force_x))
    W_friction[i] = np.sum(np.abs(force_x) * R) * dtheta_rad     # integral |T|*R dtheta [J]

exc_peak     = exc_arr[np.argmin(T_peak)]
exc_friction = exc_arr[np.argmin(W_friction)]
print('Min. peak |T|      at e = %.2f mm' % exc_peak)
print('Min. friction work at e = %.2f mm' % exc_friction)

plt.figure()
plt.plot(exc_arr, T_peak)
plt.axvline(exc_peak, color='b')
plt.text(exc_peak, np.max(T_peak), 'optimum: %.2f mm' % exc_peak,
         color='b', ha='right', va='top', rotation='vertical')
plt.xlabel('Eccentricity [mm]')
plt.ylabel('Peak transverse force $|T|$ [N]')
plt.xlim([0, 100])
plt.grid()
plt.title('Step 3: minimal peak force')
plt.show()

plt.figure()
plt.plot(exc_arr, W_friction)
plt.axvline(exc_friction, color='b')
plt.text(exc_friction, np.max(W_friction), 'optimum: %.2f mm' % exc_friction,
         color='b', ha='left', va='top', rotation='vertical')
plt.xlabel('Eccentricity [mm]')
plt.ylabel(r'Friction work $\int |T|\,R\, d\theta$ [J]')
plt.xlim([0, 100])
plt.grid()
plt.title('Step 3: minimal friction work')
plt.show()

De drukhoek- en krachtcriteria hoeven niet exact hetzelfde optimum te geven. Voor deze toepassing primeert het beperken van de piek-zijkracht tijdens het wandstrijken, omdat die stempeldoorbuiging, geleidingsslijtage en wanddiktevariatie kan veroorzaken. De wrijvingsarbeid blijft een tweede controle voor warmte en levensduur.

### Keuze van de excentriciteit

De drukhoekoptimalisatie geeft ongeveer **40,5 mm**, terwijl de piek-transversale kracht minimaal is rond **36 mm** en de wrijvingsarbeid rond **22 mm**. Omdat piek-zijkracht bij het wandstrijken het belangrijkste faalcriterium is, kiezen we de praktisch afgeronde waarde **$e=35$ mm**.

Daarmee blijft de maximale drukhoek ongeveer **26,4°**, dus onder de ontwerpgrens van 30°. Het model voorspelt een piek-transversale kracht van ongeveer **4,6 kN**. Een kleinere excentriciteit rond 22 mm zou de gemiddelde wrijving verminderen, maar geeft een hogere piek-zijkracht tijdens de arbeidsslag.

In [ ]:
# Comparison: peak transverse force and friction work -- chosen e vs centric
dtheta_rad = dtheta * np.pi / 180
print('%-18s %12s %20s' % ('', 'peak |T| [N]', 'friction work [J]'))
for e_cmp, name in [(0.0, 'centric (e=0)'), (exc, 'chosen (e=%.1f)' % exc)]:
    alpha = calculatePressureAngle(vel, e_cmp, base_radius, follower_radius, lift)
    _, _, _, _, fx, _ = calculateForce(rpm, lift, spring_constant, spring_preload,
                                       alpha, ext_load, mass, acc)
    T = np.abs(fx)                                   # transverse force |T| = |N sin(alpha)|
    R = (R0 + lift) / 1000.0                          # arm = pitch radius in m
    W = np.sum(T * R) * dtheta_rad                    # friction / transmission work per cycle [J]
    print('%-18s %12.1f %20.2f' % (name, np.max(T), W))

### Dynamische analyse: koppel, motorvermogen en vliegwiel

Door de combinatie van 120 mm slag en maximaal 25 kN proceskracht neemt het nokkoppel sterk toe. Het gemiddelde motorkoppel levert de netto arbeid per cyclus; het vliegwiel levert en absorbeert het verschil tussen gemiddeld en ogenblikkelijk koppel. Daardoor kan de motor op gemiddeld vermogen plus marge worden gekozen, terwijl nokas, tandwielkast en koppeling wel op piekkoppel worden gecontroleerd.

In [ ]:
# Torque on the cam shaft and mean motor power

M_load = normalforce_tot * np.cos(pressure_angle) * (vel / 1000.0)   # torque on cam shaft [Nm]  (= N*cos(alpha)*dS/dtheta)

M_av    = np.mean(M_load)                                       # mean torque = motor torque
P_motor = M_av * omega                                          # mean motor power [W]

print('Mean torque  M_av     = %.2f Nm' % M_av)
print('Peak torque  |M|max   = %.2f Nm' % np.max(np.abs(M_load)))
print('Mean power   P_motor  = %.1f W'  % P_motor)

plt.figure()
plt.plot(theta_deg, M_load, label=r'torque $M(\theta)$')
plt.axhline(M_av, color='r', ls='--', label=r'mean $M_{av}$')
plt.xlabel('Theta [deg]')
plt.ylabel('Torque on cam shaft [Nm]')
plt.xlim([0, 360])
plt.grid()
plt.legend()
plt.title('Torque on cam shaft')
plt.show()

### Eigen traagheid van de nok en aanvullend vliegwiel

De grotere stalen nok draagt zelf merkbaar bij aan de totale rotatietraagheid. We benaderen haar als een volle schijf met buitenstraal basiscirkel plus maximale heffing en een dikte van 50 mm. De resterende benodigde traagheid wordt door een vliegwiel geleverd. Een velgvliegwiel is bij deze schaal materiaal-efficiënter dan een volle schijf.

In [ ]:
# Flywheel design (Les 4) -- met aftrek van de eigen traagheid van de nok
dtheta_rad = dtheta * np.pi / 180
A_theta    = np.cumsum(M_av - M_load) * dtheta_rad     # arbeids-surplus A(theta) [J]
A_max      = np.max(A_theta) - np.min(A_theta)         # maximale arbeidsschommeling [J]
K          = 0.05                                      # gewenste fluctuatiecoefficient

I_req = A_max / (K * omega**2)         # vereiste TOTALE astraagheid [kg*m^2]

# Eigen rotatie-traagheid van de nok, benaderd als een VOLLE STALEN SCHIJF
# met buitenstraal = basiscirkel + maximale heffing.
rho_steel = 7850                                  # kg/m^3 (staal)
t_nok     = 0.050                                 # m, dikte (axiale breedte) van de nok -- INSTELBAAR (suggestie 20 mm)
R_nok     = (base_radius + np.max(lift)) / 1000.0 # m, buitenstraal van de nok
I_nok     = 0.5 * rho_steel * np.pi * t_nok * R_nok**4   # traagheid volle schijf [kg*m^2]

# De nok neemt een deel van de vliegwielfunctie over -> aanvullend vliegwiel:
I_flywheel = max(I_req - I_nok, 0.0)

print('Maximaal arbeids-surplus   A_max = %.2f J' % A_max)
print('Vereiste TOTALE traagheid  I_req = %.4f kg*m^2  (K = %.0f%%)' % (I_req, K*100))
print('Eigen traagheid van de nok I_nok = %.4f kg*m^2  (schijf R=%.0f mm, t=%.0f mm)' % (I_nok, R_nok*1000, t_nok*1000))
print('Aanvullend vliegwiel       I_fw  = %.4f kg*m^2  (= I_req - I_nok)' % I_flywheel)

# Ontwerp van het aanvullende vliegwiel als stalen schijf
if I_flywheel > 0:
    d = 2.00
    t = 32 * I_flywheel / (np.pi * rho_steel * d**4)
    m_fly = rho_steel * np.pi * (d/2)**2 * t
    print('Volle stalen schijf: d = %.0f mm -> dikte t = %.1f mm, massa = %.0f kg' % (d*1000, t*1000, m_fly))
    r_rim = d/2
    m_rim = I_flywheel/r_rim**2
    print('Ideale dunne velg op dezelfde diameter: massa circa %.0f kg' % m_rim)
else:
    print('De nok levert al voldoende traagheid -> geen apart vliegwiel nodig.')

plt.figure()
plt.plot(theta_deg, A_theta)
plt.xlabel('Theta [deg]')
plt.ylabel(r'Work surplus $A(\theta)$ [J]')
plt.xlim([0, 360])
plt.grid()
plt.title('Work surplus (flywheel)')
plt.show()

### Motorselectie en energie

Een exacte prijsraming is zonder leverancier, inschakelduur en aandrijflijn niet zinvol. Daarom berekenen we alleen het benodigde gemiddelde en piekasvermogen. Voor selectie wordt het gemiddelde asvermogen gedeeld door een aangenomen totaalrendement van 85% en vermenigvuldigd met een servicefactor van 1,5. De eerstvolgende standaardmotor boven die waarde is een logische kandidaat; nokas en overbrenging blijven op het veel hogere piekkoppel gedimensioneerd.

In [ ]:
# Indicative motor sizing; supplier selection is a separate engineering step
hours_year = 4000
eff_drive  = 0.85
service_factor = 1.5

P_mean = abs(M_av) * omega
P_peak = np.max(np.abs(M_load)) * omega
P_motor_required = P_mean / eff_drive * service_factor
energy_year = P_mean / eff_drive / 1000 * hours_year

print('Mean shaft power             : %7.1f kW' % (P_mean/1000))
print('Peak instantaneous shaft power: %7.1f kW' % (P_peak/1000))
print('Minimum motor rating incl. SF : %7.1f kW' % (P_motor_required/1000))
print('Indicative annual electricity : %7.0f kWh/year' % energy_year)

De berekening toont waarom de aandrijfkeuze mee moet veranderen. Het gemiddelde nokaskoppel is ongeveer **309 N·m**, maar het piekkoppel ongeveer **2,53 kN·m**. Het gemiddelde asvermogen is circa **1,94 kW** en het ogenblikkelijke piekvermogen circa **15,9 kW**.

Bij slechts 5% snelheidsvariatie is ongeveer **1068 kg·m²** totale traagheid nodig. De nok levert daarvan maar circa 6,5 kg·m². Zelfs een volle stalen schijf van 2 m diameter wordt dan ongeveer 86 mm dik en 2,1 ton zwaar; een ideale dunne velg op 1 m straal vraagt nog ongeveer 1,1 ton. Dit is geen rekenfout maar het gevolg van veel vormarbeid in een deel van één trage omwenteling. Voor een echte blikjeslijn is daarom een bestaande persarchitectuur met een groot centraal vliegwiel, meerdere stations, een krukaandrijving of een regeneratieve servo logischer dan één afzonderlijke open nok.

---

## Antwoord op de examenvraag: aanpassing voor een drankblikje

Voor een drankblikje volstaat het niet om alleen de heffing en kracht te vergroten. De volledige aandrijfketen moet mee veranderen:

| onderdeel | aanpassing en reden |
|---|---|
| slag en timing | Totale slag naar 120 mm. De arbeidsslag krijgt 130° en de terugkeer 120° om versnelling en krachtpieken te beperken; 50° blijft over voor transfer. |
| proceskracht | Ontwerpschatting 5 kN bij redraw, oplopend tot 25 kN bij wandstrijken en 3 kN bij strippen. Definitief bepalen uit materiaal, reductie, wrijving en smering. |
| bewegingswet | Cycloïde behouden wegens nul snelheid en versnelling aan de grenzen. Bij zeer hoge snelheid een 7e-orde of servoprofiel onderzoeken. |
| nok | Steekstraal 250 mm en basiscirkel 200 mm: een veel grotere nok houdt de drukhoek bij de grotere slag beheersbaar. Ook nokbreedte, materiaal en oppervlakteharding moeten toenemen. |
| rolvolger | Rolstraal 50 mm met zwaar rollager. Groter contactoppervlak verlaagt de contactspanning; ondersnijding en kromtestraal blijven bepalend. |
| excentriciteit | 35 mm om de drukhoek en zijkracht tijdens de zwaarste arbeidsslag te verlagen. De terugkeerdrukhoek moet onder 30° blijven. |
| volger, ram en frame | Equivalente massa circa 40 kg; stijvere geleidingen, dikkere stempel en stijver frame om doorbuiging en wanddiktevariatie te beperken. |
| veer/retour | Voorspanning 2 kN en herberekende veerconstante met 25% marge. Voor industriële betrouwbaarheid liever positieve retour via conjugate cam, groefnok of kruk. |
| motor en overbrenging | Groter gemiddeld vermogen en vooral veel hoger piekkoppel. Motor op gemiddeld vermogen met servicefactor; as, reductiekast en koppeling op piekkoppel. |
| vliegwiel | Veel grotere benodigde traagheid wegens de geconcentreerde vormarbeid. Een groot velgvliegwiel is efficiënter dan een kleine volle schijf. |
| matrijs en proces | Meerdere strijkringen, kleine reductie per ring, goede smering/koeling, geleide stripper en controle van wanddikte en scheurvorming. |

**Kort examenantwoord.** Voor de overgang naar een drankblikje vergroten we de totale heffing van 20 naar ongeveer 120 mm en de proceskracht van ongeveer 1 kN naar een aangenomen piek van 25 kN. We spreiden de arbeidsslag over een grotere nokhoek en verlagen het toerental naar 60 rpm om inertiekrachten te beperken. De grotere slag vereist een grotere nok (steekstraal 250 mm), een grotere rolvolger (50 mm), stijvere ram en geleidingen en een sterkere retourvoorziening. De excentriciteit wordt verhoogd om de drukhoek tijdens het zwaar belaste wandstrijken klein te houden. Door het hogere koppel zijn ook een grotere motor, zwaardere as en lagering en aanzienlijk meer vliegwieltraagheid nodig. Voor echte massaproductie zou ik de open nok met veer vervangen door een positief gesloten nok, krukpers of servopers, omdat die de grote vorm- en retourkrachten betrouwbaarder overdraagt.

### Berekende controlewaarden

| controle | resultaat |
|---|---:|
| maximale drukhoek bij $e=35$ mm | 26,4° |
| minimale positieve kromtestraal steekkromme | 186,6 mm |
| gekozen rolstraal | 50 mm, dus geen ondersnijding |
| berekende minimale veerconstante | 112,6 N/mm |
| gekozen veerconstante met 25% marge | 140,7 N/mm |
| minimale / maximale contactkracht | 0,19 kN / 44,1 kN |
| piek-transversale kracht | 4,6 kN |
| gemiddeld / piekkoppel | 309 N·m / 2,53 kN·m |
| gemiddeld / piekasvermogen | 1,94 kW / 15,9 kW |
| vereiste totale traagheid bij $K=5\%$ | 1068 kg·m² |